# Investment Suggestion Agent with RAG

Same agent workflow as `investment_agent.ipynb` but powered by **Retrieval Augmented Generation** using ChromaDB as an in-memory vector store.

## Architecture
```
docs/ (articles) → Chunked → ChromaDB (in-memory vector DB)
                                    ↓
State: { name, age, income, investment_amount, industry, companies[], research[], summary }

User Research  →  Market Research  →  Company Research  →  Summarise
     ↑________________↑____________________↑
              (loop back if needed)

Tools: RAG Search (ChromaDB), Calculator, Coding, Database
```

## Sample Profile
- **Name**: Praveen | **Age**: 60 | **Income**: ₹5L/month
- **Investment**: ₹50,000 | **Industry**: Food Industry

In [13]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "langchain", "langchain-openai", "langchain-chroma", "langchain-text-splitters",
    "langgraph", "chromadb", "pydantic", "python-dotenv"])
print("✅ All packages installed")

✅ All packages installed



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [14]:
import json
import sqlite3
import math
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional

llm = ChatOpenAI(model="gpt-4o", temperature=0.3)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("✅ Imports done")

✅ Imports done


## Step 1: Load Documents & Build ChromaDB Vector Store

In [15]:
# ── Load all .txt docs from the docs/ folder ─────────────────────────────────
docs_path = Path("docs")
raw_docs = []

for file_path in sorted(docs_path.glob("*.txt")):
    content = file_path.read_text()
    raw_docs.append(Document(
        page_content=content,
        metadata={"source": file_path.name}
    ))
    print(f"  📄 Loaded: {file_path.name} ({len(content)} chars)")

print(f"\n✅ Loaded {len(raw_docs)} documents")

  📄 Loaded: dairy_segment_deep_dive.txt (2211 chars)
  📄 Loaded: food_industry_overview_2026.txt (1839 chars)
  📄 Loaded: food_sector_risks_2026.txt (2607 chars)
  📄 Loaded: investment_strategies_seniors.txt (2247 chars)
  📄 Loaded: top_food_companies_india.txt (2988 chars)

✅ Loaded 5 documents


In [16]:
# ── Chunk documents ──────────────────────────────────────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " "]
)

chunks = text_splitter.split_documents(raw_docs)
print(f"✅ Split into {len(chunks)} chunks")
print(f"   Sample chunk: {chunks[0].page_content[:100]}...")

✅ Split into 32 chunks
   Sample chunk: Title: Dairy Segment — India's Largest Food Sub-Sector (Deep Dive 2026)

Market Size: $123 billion (...


In [ ]:
# ── Build ChromaDB in-memory vector store ────────────────────────────────────
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="investment_docs"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Quick test
test_results = retriever.invoke("cake making companies in india")
print(f"✅ ChromaDB ready — {vectorstore._collection.count()} vectors stored")
print(f"   Test query returned {len(test_results)} results")
print(f"   Top result from: {test_results[0].metadata['source']}")


✅ ChromaDB ready — 224 vectors stored
   Test query returned 5 results
   Top result from: top_food_companies_india.txt
   Top Vector from: page_content='Title: Top Listed Food Companies in India — Financial Snapshot (Q1 2026)

1. NESTLE INDIA (NSE: NESTLEIND)
   - Market Cap: ₹2,45,000 Crore
   - P/E Ratio: 72.5
   - Revenue Growth (YoY): 9.2%
   - Net Profit Margin: 14.8%
   - Dividend Yield: 1.4%
   - Key Brands: Maggi, Nescafe, KitKat, Milkmaid
   - Analyst Rating: HOLD — expensive valuation but strong brand moat
   - 5-Year Return: 78% (CAGR ~12.2%)' metadata={'source': 'top_food_companies_india.txt'}


## Step 2: Define State & Tools

In [18]:
# ── Agent State ──────────────────────────────────────────────────────────────
class InvestmentState(TypedDict):
    name: str
    age: int
    monthly_income: float
    investment_amount: float
    industry: str

    user_profile_summary: Optional[str]
    companies: list[str]
    research: list[str]
    summary: str

    next_node: Optional[str]
    iteration: int

print("✅ State defined")

✅ State defined


In [19]:
# ── Tools ────────────────────────────────────────────────────────────────────

# 1. RAG Search Tool — replaces web search with ChromaDB retrieval
@tool
def rag_search(query: str) -> str:
    """Search the investment knowledge base for information about food industry companies, market trends, risks, and investment strategies."""
    results = retriever.invoke(query)
    if not results:
        return "No relevant documents found."
    output = []
    for i, doc in enumerate(results, 1):
        print(doc)
        output.append(f"[Source: {doc.metadata['source']}]\n{doc.page_content}")
    return "\n\n---\n\n".join(output)


# 2. Calculator Tool
@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression for financial calculations. Examples: '50000 * 0.12', '50000 * (1.14)**3'"""
    try:
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return str(result)
    except Exception as e:
        return f"Calculation error: {e}"


# 3. Coding Tool
@tool
def run_python(code: str) -> str:
    """Execute Python code for data analysis. Store final answer in a variable named 'result'."""
    try:
        namespace = {"math": math, "json": json}
        exec(code, namespace)
        return str(namespace.get("result", "Code executed successfully"))
    except Exception as e:
        return f"Code error: {e}"


# 4. Database Tool
_db = sqlite3.connect(":memory:")
_db.execute("""
    CREATE TABLE investment_data (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        category TEXT,
        key TEXT,
        value TEXT
    )
""")
_db.commit()

@tool
def database_query(sql: str) -> str:
    """Run SQL against investment_data(id, category, key, value). Use INSERT to save, SELECT to retrieve."""
    try:
        cur = _db.execute(sql)
        _db.commit()
        rows = cur.fetchall()
        return json.dumps(rows) if rows else "Query executed successfully"
    except Exception as e:
        return f"DB error: {e}"


TOOLS = [rag_search, calculator, run_python, database_query]
llm_with_tools = llm.bind_tools(TOOLS)

print("✅ Tools defined: rag_search, calculator, run_python, database_query")

✅ Tools defined: rag_search, calculator, run_python, database_query


## Step 3: Agent Nodes

In [20]:
# ── Helper: run an agent node with tool loop ────────────────────────────────
def run_agent_node(system_prompt: str, user_message: str, tools: list) -> str:
    from langchain_core.messages import AIMessage, ToolMessage

    tool_map = {t.name: t for t in tools}
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_message)
    ]

    for _ in range(10):
        response: AIMessage = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for tc in response.tool_calls:
            tool_fn = tool_map.get(tc["name"])
            if tool_fn:
                result = tool_fn.invoke(tc["args"])
                messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return messages[-1].content if hasattr(messages[-1], "content") else "Agent finished"

print("✅ Agent runner defined")

✅ Agent runner defined


In [21]:
# ── Node 1: User Research ───────────────────────────────────────────────────
def user_research(state: InvestmentState) -> dict:
    print(f"\n{'='*50}")
    print("🔍 NODE: User Research")
    print(f"{'='*50}")

    system_prompt = """
You are a certified financial planner. Analyse the client's profile to determine:
1. Risk appetite (conservative / moderate / aggressive)
2. Investment goals (growth, income, capital preservation)
3. Suitable investment horizon
4. Recommended allocation strategy

Use rag_search to find relevant investment strategy guidelines for this age group.
Use calculator for financial ratios and return estimates.
Save key findings to the database.
"""

    user_message = f"""
Client profile:
- Name: {state['name']}
- Age: {state['age']}
- Monthly income: ₹{state['monthly_income']:,.0f}
- Investment amount: ₹{state['investment_amount']:,.0f}
- Target industry: {state['industry']}

Analyse this profile and produce a concise investment profile summary.
"""

    profile = run_agent_node(system_prompt, user_message, TOOLS)
    print(f"\n{profile}")

    return {
        "user_profile_summary": profile,
        "next_node": "market_research",
        "iteration": state.get("iteration", 0)
    }


# ── Node 2: Market Research ─────────────────────────────────────────────────
def market_research(state: InvestmentState) -> dict:
    print(f"\n{'='*50}")
    print("📊 NODE: Market Research")
    print(f"{'='*50}")

    system_prompt = """
You are a market research analyst. Using the knowledge base:
1. Search for current trends and outlook for the target industry
2. Identify top-performing segments
3. Assess market risks
4. Suggest 3-5 specific companies to research further

Use rag_search to retrieve market data and company information.
Save company names to the database.
At the end, output: NEXT_NODE: company_research (or NEXT_NODE: user_research if more info needed)
"""

    user_message = f"""
User profile:\n{state['user_profile_summary']}

Target industry: {state['industry']}
Investment amount: ₹{state['investment_amount']:,.0f}

Research the {state['industry']} sector and recommend companies.
"""

    result = run_agent_node(system_prompt, user_message, TOOLS)
    print(f"\n{result}")

    next_node = "company_research"
    if "NEXT_NODE: user_research" in result:
        next_node = "user_research"

    research = state.get("research", [])
    research.append(f"[Market Research]\n{result}")

    return {
        "research": research,
        "companies": state.get("companies", []),
        "next_node": next_node,
        "iteration": state.get("iteration", 0) + 1
    }


# ── Node 3: Company Research ────────────────────────────────────────────────
def company_research(state: InvestmentState) -> dict:
    print(f"\n{'='*50}")
    print("🏢 NODE: Company Research")
    print(f"{'='*50}")

    system_prompt = """
You are an equity research analyst. For each company:
1. Search for financials (revenue growth, P/E ratio, market cap)
2. Check analyst ratings and recent sentiment
3. Calculate potential returns using calculator
4. Rate each: BUY / HOLD / AVOID with reasoning
5. Suggest allocation of the investment amount across top picks

Use rag_search to retrieve company-specific data from the knowledge base.
At the end, output: NEXT_NODE: summarise (or NEXT_NODE: market_research if more data needed)
"""

    market_notes = "\n".join(state.get("research", [])[-2:])

    user_message = f"""
User profile:\n{state['user_profile_summary']}

Market research notes:\n{market_notes}

Investment amount: ₹{state['investment_amount']:,.0f}
Industry: {state['industry']}

Research companies and provide ratings with allocation suggestions.
"""

    result = run_agent_node(system_prompt, user_message, TOOLS)
    print(f"\n{result}")

    next_node = "summarise"
    if "NEXT_NODE: market_research" in result:
        next_node = "market_research"

    research = state.get("research", [])
    research.append(f"[Company Research]\n{result}")

    return {
        "research": research,
        "next_node": next_node,
        "iteration": state.get("iteration", 0) + 1
    }


# ── Node 4: Summarise ───────────────────────────────────────────────────────
def summarise(state: InvestmentState) -> dict:
    print(f"\n{'='*50}")
    print("📋 NODE: Summarise")
    print(f"{'='*50}")

    system_prompt = """
You are a senior investment advisor. Compile all research into a final report:
1. Client profile snapshot
2. Market outlook
3. Top investment picks with ratings
4. Suggested allocation table
5. Risk warnings
6. Expected returns (conservative / moderate / optimistic)

Use calculator for final return projections.
Format clearly with sections and bullet points.
"""

    all_research = "\n\n".join(state.get("research", []))

    user_message = f"""
Client: {state['name']}, Age: {state['age']}
Investment: ₹{state['investment_amount']:,.0f} in {state['industry']}

User Profile:\n{state['user_profile_summary']}

All Research:\n{all_research}

Produce the final investment recommendation report.
"""

    report = run_agent_node(system_prompt, user_message, TOOLS)
    return {"summary": report}


print("✅ All 4 agent nodes defined")

✅ All 4 agent nodes defined


## Step 4: Build LangGraph with Routing

In [22]:
# ── Routing Logic ───────────────────────────────────────────────────────────
def route_market(state: InvestmentState) -> str:
    if state.get("next_node") == "user_research" and state.get("iteration", 0) < 2:
        print("↩️  Routing back to User Research")
        return "user_research"
    print("➡️  Routing to Company Research")
    return "company_research"


def route_company(state: InvestmentState) -> str:
    if state.get("next_node") == "market_research" and state.get("iteration", 0) < 3:
        print("↩️  Routing back to Market Research")
        return "market_research"
    print("➡️  Routing to Summarise")
    return "summarise"


# ── Build Graph ──────────────────────────────────────────────────────────────
graph = StateGraph(InvestmentState)

graph.add_node("user_research", user_research)
graph.add_node("market_research", market_research)
graph.add_node("company_research", company_research)
graph.add_node("summarise", summarise)

graph.add_edge(START, "user_research")
graph.add_edge("user_research", "market_research")

graph.add_conditional_edges(
    "market_research", route_market,
    {"user_research": "user_research", "company_research": "company_research"}
)
graph.add_conditional_edges(
    "company_research", route_company,
    {"market_research": "market_research", "summarise": "summarise"}
)

graph.add_edge("summarise", END)

investment_agent = graph.compile()

print("✅ RAG Investment Agent compiled!")
print("Flow: User Research → Market Research → Company Research → Summarise")
print("Tools: rag_search (ChromaDB), calculator, run_python, database_query")

✅ RAG Investment Agent compiled!
Flow: User Research → Market Research → Company Research → Summarise
Tools: rag_search (ChromaDB), calculator, run_python, database_query


## Step 5: Run the Agent

In [23]:
initial_state: InvestmentState = {
    "name": "Praveen",
    "age": 60,
    "monthly_income": 500000,
    "investment_amount": 50000,
    "industry": "Food Industry",
    "user_profile_summary": None,
    "companies": [],
    "research": [],
    "summary": "",
    "next_node": None,
    "iteration": 0
}

print("🚀 Starting RAG Investment Agent for Praveen...")
print(f"   Age: {initial_state['age']} | Income: ₹{initial_state['monthly_income']:,.0f}/month")
print(f"   Investment: ₹{initial_state['investment_amount']:,.0f} | Industry: {initial_state['industry']}")
print(f"   Knowledge Base: {vectorstore._collection.count()} vectors in ChromaDB")
print()

result = investment_agent.invoke(initial_state)

🚀 Starting RAG Investment Agent for Praveen...
   Age: 60 | Income: ₹500,000/month
   Investment: ₹50,000 | Industry: Food Industry
   Knowledge Base: 192 vectors in ChromaDB


🔍 NODE: User Research
page_content='Recommended Asset Allocation for Age 60:
- Fixed Income / Debt: 50-60% (FDs, bonds, debt mutual funds)
- Large-Cap Equity: 20-25% (blue-chip stocks, index funds)
- Dividend-Yield Stocks: 10-15% (companies with consistent payouts)
- Gold / Alternative: 5-10%
- Cash / Liquid: 5%' metadata={'source': 'investment_strategies_seniors.txt'}
page_content='Recommended Asset Allocation for Age 60:
- Fixed Income / Debt: 50-60% (FDs, bonds, debt mutual funds)
- Large-Cap Equity: 20-25% (blue-chip stocks, index funds)
- Dividend-Yield Stocks: 10-15% (companies with consistent payouts)
- Gold / Alternative: 5-10%
- Cash / Liquid: 5%' metadata={'source': 'investment_strategies_seniors.txt'}
page_content='Recommended Asset Allocation for Age 60:
- Fixed Income / Debt: 50-60% (FDs, bonds, deb

In [24]:
print("\n" + "="*60)
print("    INVESTMENT RECOMMENDATION REPORT (RAG)")
print("="*60)
print(result["summary"])
print("="*60)


    INVESTMENT RECOMMENDATION REPORT (RAG)
**Final Investment Recommendation Report for Praveen**

---

### 1. Client Profile Snapshot

- **Name:** Praveen
- **Age:** 60
- **Investment Amount:** ₹50,000
- **Risk Appetite:** Conservative
- **Investment Goals:** Income and Capital Preservation
- **Investment Horizon:** 5 years

### 2. Market Outlook

- **Food Industry Growth:** Projected to reach $535 billion by 2026, growing at a CAGR of 11-12%.
- **Key Segments:** Dairy Products, Grains & Cereals, Meat & Poultry, Fruits & Vegetables Processing, and Packaged Foods & Snacks.
- **Market Risks:** Commodity price volatility and climate impacts affecting raw material costs.

### 3. Top Investment Picks with Ratings

- **Nestle India (NSE: NESTLEIND)**
  - **Market Cap:** ₹2,45,000 Crore
  - **Analyst Rating:** HOLD
  - **5-Year Return:** 78% (CAGR ~12.2%)
  - **Reasoning:** Strong brand presence with stable returns but high valuation.

- **Britannia Industries (NSE: BRITANNIA)**
  - **Marke